In [107]:
"""
<라우팅(Routing) 기법>
1. 병렬(Parallel) 처리하는 기법
  - 하나의 입력(프롬프트)으로 두 가지 작업(두개 체인)을 동시에 처리하고자 할 때 사용

2. 조건부 라이팅 분기 처리 기법
  - 사용자의 입력 의도에 따라 A체인 또는 B체인으로 분기(선택)하여 처리하고자 할 때 사용
"""

'\n<라우팅(Routing) 기법>\n1. 병렬(Parallel) 처리하는 기법\n  - 하나의 입력(프롬프트)으로 두 가지 작업(두개 체인)을 동시에 처리하고자 할 때 사용\n\n2. 조건부 라이팅 분기 처리 기법\n  - 사용자의 입력 의도에 따라 A체인 또는 B체인으로 분기(선택)하여 처리하고자 할 때 사용\n'

#### 사용할 라이브러리

In [108]:
### LLM 기본 라이브러리
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

### 병렬처리에 사용되는 라이브러리
# - 입력 데이터를 수정 없이 원본 그대로 다음 단계(체인)으로 전달하는 라이브러리
from langchain_core.runnables import RunnablePassthrough

# - 여러 체인을 동시에 병렬로 실행하는 라이브러리
from langchain_core.runnables import RunnableParallel

# - 조건부 라우팅 분기처리 라이브러리
from langchain_core.runnables import RunnableBranch

### LLM 기본 틀(프롬프트, 모델, 출력) 정의

In [109]:
"""
- 사용모델 : mistral, 창의성 0.1, 응답 길이 100, 변수명 llm_model
- 출력파서 : 변수명 output_parser
"""

'\n- 사용모델 : mistral, 창의성 0.1, 응답 길이 100, 변수명 llm_model\n- 출력파서 : 변수명 output_parser\n'

In [110]:
# - 사용모델 : mistral, 창의성 0.1, 응답 길이 100, 변수명 llm_model
llm_model = ChatOllama(
    model = "mistral",
    temperature = 0.1,
    max_tokens = 100
)

# - 출력파서 : 변수명 output_parser
output_parser = StrOutputParser()

### 병렬 처리 프롬프트 및 체인 구성

In [111]:
"""
- 프롬프트 1 : 제품의 [장점]을 분석하는 프롬프트
- 프롬프트 2 : 제품의 [단점]을 분석하는 프롬프트
"""

'\n- 프롬프트 1 : 제품의 [장점]을 분석하는 프롬프트\n- 프롬프트 2 : 제품의 [단점]을 분석하는 프롬프트\n'

#### - 프롬프트 1 : 제품의 [장점]을 분석하는 프롬프트 및 체인 구성

In [112]:
### 시스템(챗봇)에게 역할(페르소나) 부여
# - 사용자 질문 : product에 담겨져서 전달 받음
# - {product}의 값은 key가 됨.
request_msg = "{product}의 가장 큰 장점 2가지만 작성해줘."

### from_template() 프롬프트 템플릿 함수 사용
# - 변수명 : pro_prompt
pro_prompt = ChatPromptTemplate.from_template(request_msg)
pro_prompt

ChatPromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, template='{product}의 가장 큰 장점 2가지만 작성해줘.'), additional_kwargs={})])

In [113]:
### 제품의 장점을 분석하는 체인 구성
# - 체인 변수명 : pro_chain
pro_chain = pro_prompt | llm_model | output_parser
pro_chain

ChatPromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, template='{product}의 가장 큰 장점 2가지만 작성해줘.'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', temperature=0.1)
| StrOutputParser()

#### - 프롬프트 2 : 제품의 [단점]을 분석하는 프롬프트

In [114]:
### 시스템(챗봇)에게 역할(페르소나) 부여
# - 사용자 질문 : product에 담겨져서 전달 받음
#  -> request_msg 1회성으로 쓰는 변수라 위와 똑같이 써도 상관은 없음.
request_msg = "{product}의 구매를 망설이게 하는 단점 1가지만 작성해줘."

### from_template() 프롬프트 템플릿 함수 사용
# - 변수명 : con_prompt
con_prompt = ChatPromptTemplate.from_template(request_msg)
con_prompt

### 제품의 단점을 분석하는 체인 구성
# - 체인 변수명 : con_chain
con_chain = con_prompt | llm_model | output_parser
con_chain

ChatPromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, template='{product}의 구매를 망설이게 하는 단점 1가지만 작성해줘.'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', temperature=0.1)
| StrOutputParser()

### [장점]과 [단점] 체인(chain)을 병렬로 구성하기(병렬로 묶기)

In [115]:
### 입력된 질문(product)이 두 체인에 동시에 전달되도록 병렬로 묶기
# - RunnableParallel 클래스 : 병렬처리(두개 체인 묶기) 클래스
#   -> 사용되는 속성명은 작성자가 직접 작명함(체인 특성에 맞게 변수명 작명)
#   -> 속성(변수)명은 key가 됨(딕셔너리의 key 의미를 가짐)
parallel_analysis = RunnableParallel(
    ### 장점 정의(key = value)
    pros = pro_chain,

    ### 단점 정의(key = value)
    cons = con_chain
)
parallel_analysis

{
  pros: ChatPromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, template='{product}의 가장 큰 장점 2가지만 작성해줘.'), additional_kwargs={})])
        | ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', temperature=0.1)
        | StrOutputParser(),
  cons: ChatPromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, template='{product}의 구매를 망설이게 하는 단점 1가지만 작성해줘.'), additional_kwargs={})])
        | ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', temperature=0.1)
        | StrOutputParser()
}

### 사용자의 입력(product)을 원본 그대로 유지한 채로 병렬 묶음에 전달

In [116]:
### RunnablePassthrough 클래스 : 원본 데이터를 그대로 유지한채 전달하는 클래스
# - assign() 함수 : 원형 그대로 parallel_analysis 병렬처리 체인들에게 전달하는 기능
# - assign(analysis_result = parallel_analysis) : analysis_result 변수명은 수정가능(딕셔너리 key로 사용됨)
# - 각 병렬 처리 결과값을 analysis_result 변수(key)가 관리함
start_chain = RunnablePassthrough.assign(analysis_result = parallel_analysis)
start_chain

RunnableAssign(mapper={
  analysis_result: {
                     pros: ChatPromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, template='{product}의 가장 큰 장점 2가지만 작성해줘.'), additional_kwargs={})])
                           | ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', temperature=0.1)
                           | StrOutputParser(),
                     cons: ChatPromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, template='{product}의 구매를 망설이게 하는 단점 1가지만 작성해줘.'), additional_kwargs={})])
                           | ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', tem

### 병렬 체인 실행시키기(start_chain을 이용해서 시작 실행)

In [117]:
### 사용자의 질문 정의 : "드럼 세탁기"
question = "드럼 세탁기"
# - start_chain에게 사용자의 질문 전달하여 체인 실행 시키기
# - 최종 결과값을 받을 변수명 : parallel_result
# - invoke({"product" : question})이 start_chain에 넘겨줌 -> parallel_analysis에 넘겨줌 -> pro냐 con에 따라 프롬프트로 전달됨.
# - invoke({{프롬프트에 정의한 변수 key : 사용자가 입력한 질문})
parallel_result = start_chain.invoke({"product" : question})
parallel_result

{'product': '드럼 세탁기',
 'analysis_result': {'pros': ' 1. 효율적인 세척: 드럼 세탁기는 많은 양의 옷을 한 번에 세울 수 있어 시간 및 수력을 절약할 수 있습니다. 또한 강력한 세탁 능력으로 옷들을 깨끗하게 세칩니다.\n\n2. 편리함: 드럼 세탁기는 집에서 옷을 세울 때 매우 편리합니다. 세탁 기계를 사용하면 옷을 세울 필요가 없고, 세탁 기계가 작동하면 옷이 깨끗해지는 것을 볼 수 있습니다. 또한 세탁 기계를 사용하면 옷이 손상되는 것을 방지할 수 있습니다.',
  'cons': ' 1. 비용 문제: 드럼 세탁기는 상당한 비용을 요구하며, 일반적인 세탁기와 비교하여 가격이 높습니다. 또한 유지 보수와 에너지 소비 등 운영 비용도 고려해야 합니다. 따라서 가구 세트에 드럼 세탁기를 포함시키는 것은 일반적인 세탁기로 대체할 수 있는 경우 비용 문제로 인해 망설이게 할 수 있습니다.'}}

In [153]:
"""
<결과값을 이용하여 출력하기>
- 사용자 질문 출력
- 장점 출력 : 각각 분리하여 출력
- 단점 출력
"""
result_pros = parallel_result['analysis_result']['pros'].split("\n\n")


# - 사용자 질문 출력
print(f"[사용자의 질문] : {parallel_result['product']}")
print("="*100)
# - 장점 출력
[print(f"{parallel_result['product']}에 대한 [장점] : {i.strip()}") for i in result_pros]
print("="*100)
# - 단점 출력
print(f"{parallel_result['product']}에 대한 [단점] : {parallel_result['analysis_result']['cons'].strip()}")

[사용자의 질문] : 드럼 세탁기
드럼 세탁기에 대한 [장점] : 1. 효율적인 세척: 드럼 세탁기는 많은 양의 옷을 한 번에 세울 수 있어 시간 및 수력을 절약할 수 있습니다. 또한 강력한 세탁 능력으로 옷들을 깨끗하게 세칩니다.
드럼 세탁기에 대한 [장점] : 2. 편리함: 드럼 세탁기는 집에서 옷을 세울 때 매우 편리합니다. 세탁 기계를 사용하면 옷을 세울 필요가 없고, 세탁 기계가 작동하면 옷이 깨끗해지는 것을 볼 수 있습니다. 또한 세탁 기계를 사용하면 옷이 손상되는 것을 방지할 수 있습니다.
드럼 세탁기에 대한 [단점] : 1. 비용 문제: 드럼 세탁기는 상당한 비용을 요구하며, 일반적인 세탁기와 비교하여 가격이 높습니다. 또한 유지 보수와 에너지 소비 등 운영 비용도 고려해야 합니다. 따라서 가구 세트에 드럼 세탁기를 포함시키는 것은 일반적인 세탁기로 대체할 수 있는 경우 비용 문제로 인해 망설이게 할 수 있습니다.


### 조건부 라우팅(Routing) 분기 처리

In [ ]:
"""
1. 사용자의 질문 의도를 판별하여 의도에 맞는 체인 실행
  - [의도 예시] : 사용자의 질문 내용을 확인하여
               : 긍정 또는 부정 의도에 따른 긍정체인과 부정체인 선별 실행
"""

In [158]:
### 챗봇(시스템) 페르소나(역할) 프롬프트 정의하기
# - 역할 정의 : "다음 리뷰가 긍정적이면 'POSITIVE', 부정적이면 'NEGATIVE'라고 대답해 : {review}"
request_msg = "다음 리뷰가 긍정적이면 'POSITIVE', 부정적이면 'NEGATIVE'라고 대답해 : {review}"
request_msg

# - 모델 정의 하기 변수명 : llm
llm = ChatOllama(
    model = "mistral",
    temperature = 0.1,
    max_tokens = 100
)
llm

# - 프롬프트 변수명 : routing_prompt
routing_prompt = ChatPromptTemplate.from_template(request_msg)
routing_prompt

# - 출력 파서 정의
parser = StrOutputParser()
parser

# - 체인 구성 : 변수명은 routing_chain
routing_chain = routing_prompt | llm | parser
routing_chain

ChatPromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, template="다음 리뷰가 긍정적이면 'POSITIVE', 부정적이면 'NEGATIVE'라고 대답해 : {review}"), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', temperature=0.1)
| StrOutputParser()

#### [긍정]일때의 프롬프트 및 체인 구성

In [163]:
### 긍정일 때 실행할 프롬프트 및 체인
# - 프롬프트   : "매우 기쁜 어조로 다음 리뷰에 대한 감사 답글을 작성해 : {review}"
#             : 변수명 : positive_prompt
positive_prompt = ChatPromptTemplate.from_template("매우 기쁜 어조로 다음 리뷰에 대한 감사 답글을 작성해 : {review}")
positive_prompt
# - 체인변수명 : positive_chain
positive_chain = positive_prompt | llm | parser
positive_chain

ChatPromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, template='매우 기쁜 어조로 다음 리뷰에 대한 감사 답글을 작성해 : {review}'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', temperature=0.1)
| StrOutputParser()

#### [부정]일때의 프롬프트 및 체인 구성

In [161]:
### 부정일 때 실행할 프롬프트 및 체인
# - 프롬프트   : "매우 죄송한 어조로 다음 리뷰에 대한 사과 답글을 작성해 : {review}"
#             : 변수명 : negative_prompt
negative_prompt = ChatPromptTemplate.from_template("매우 죄송한 어조로 다음 리뷰에 대한 사과 답글을 작성해 : {review}")
negative_prompt
# - 체인변수명 : negative_chain
negative_chain = negative_prompt | llm | parser
negative_chain

ChatPromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, template='매우 죄송한 어조로 다음 리뷰에 대한 사과 답글을 작성해 : {review}'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', temperature=0.1)
| StrOutputParser()

#### 의도를 판별하는 조건 처리(조건부 라우팅 처리)

In [164]:
### RunnableBranch에서 조건으로 사용할 함수 정의
# - routing_chain 실행하여 결과값 확인
#   -> 긍정이면 True, 부정이면 False를 반환해 주는 함수 정의하여 사용
# - x : 사용자 질문(review)가 들어있는 딕셔너리 변수
#     : RunnableBranch 내부에서 정의된 약속된 변수명 입니다.
def is_positive(x) :
    ### routing_chain을 독립적으로 호출하여 실행
    # - x 매개변수에는 사용자가 질문한 review라는 key의 값이 들어있음
    #   -> x = {"review" : "사용자 질문"}}
    rout = routing_chain.invoke({"review" : x["review"]})

    print(f"routing_chain 결과값 ===> {rout}")
    ### routing_chain의 실행결과가 "POSITIVE" 인지 아닌지 확인 후 반환
    return "POSITIVE" in rout

In [165]:
### 의도를 판별(조건처리)하는 라우팅 클래스(RunnableBranch) 사용
# - RunnableBranch 작성 문법
"""
    RunnableBranch(
        # 조건(True 또는 False)이 참일 때(의도가 긍정일 때) : if
        (조건, 긍정 체인),

        # 조건이 거짓일 때(의도가 부정일 때) : else
        부정 체인
    )
"""

branch_chain = RunnableBranch(
    # 조건(True 또는 False)이 참일 때(의도가 긍정일 때) : if
    #  - 조건은 함수(is_positive)로 정의하여 처리 : routing_chain 실행하여 결과값 확인
    #   -> 긍정이면 True, 부정이면 False를 반환해주는 함수 정의하여 사용
    (is_positive, positive_chain),

    # 조건이 거짓일 때(의도가 부정일 때) : else
    negative_chain
)
branch_chain

RunnableBranch(branches=[(RunnableLambda(is_positive), ChatPromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, template='매우 기쁜 어조로 다음 리뷰에 대한 감사 답글을 작성해 : {review}'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', temperature=0.1)
| StrOutputParser())], default=ChatPromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, template='매우 죄송한 어조로 다음 리뷰에 대한 사과 답글을 작성해 : {review}'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', temperature=0.1)
| StrOutputParser())

### 오류(예외)를 대비한 대체 체인(Fallback 체인) 정의

In [167]:
### 체인 실행 중에 API 타임아웃 등의 예외가 발생하는 경우
# -> 별도의 답변을 제공하는 체인으로 우회 시키는 방법을 "FallBack" 이라고 칭함
# -> FallBack 처리 클래스 : RunnableLambda 클래스 사용
# -> FallBack 처리 함수 : with_fallbacks(RunnableLambda 클래스)

### RunnableLambda 라이브러리 정의
from langchain_core.runnables import RunnableLambda

### 예외 발생시 작동시킬 체인 클래스 생성
fallback_chain = RunnableLambda(lambda x: "일시적인 오류로 답변을 생성할 수 없습니다. 고객센터로 문의해 주세요!!")

### Fallback 상태 확인 함수 정의
# - 체인 작동 중 예외가 발생했는지 상태확인을 시작하는 함수
# - 예외가 발생하면 자동으로 fallback_chain을 작동시킴
safe_branch_chain = branch_chain.with_fallbacks([fallback_chain])
safe_branch_chain

RunnableWithFallbacks(runnable=RunnableBranch(branches=[(RunnableLambda(is_positive), ChatPromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, template='매우 기쁜 어조로 다음 리뷰에 대한 감사 답글을 작성해 : {review}'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', temperature=0.1)
| StrOutputParser())], default=ChatPromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, template='매우 죄송한 어조로 다음 리뷰에 대한 사과 답글을 작성해 : {review}'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, model='mistral', temperature=0.1)
| StrOutputParser()), fallbacks=[RunnableLambda(lambda x

### 테스트 하기

In [172]:
test_review = "제품이 마음에 들지 않아요."
print(f"[고객 리뷰] : {test_review}")

### 자동으로 긍정/부정 라우팅되어 사과문 작성하기
rout_res = safe_branch_chain.invoke({"review" : test_review})
print(f"[답글] : {rout_res}")

[고객 리뷰] : 제품이 마음에 들지 않아요.
routing_chain 결과값 ===>  NEGATIVE
[답글] :  안녕하세요! 제가 당신의 리뷰에 대한 감사를 표합니다. 당신의 의견은 우리 회사에 매우 중요합니다. 제품이 당신의 기대를 충족하지 못했던 것 같아 죄송합니다.

우리 회사는 고객의 만족을 최우선으로 생각하며, 당신의 피드백을 통해 더 나은 제품을 만들기 위해 노력하고 있습니다. 당신의 문제점을 확인하고 개선하기 위해 연락을 드리겠습니다. 또한, 당신의 문제점을 해결하기 위해 추가적인 지원을 제공할 수 있는지 확인하겠습니다.

또한, 당신의 리뷰를 통해 우리 회사는 더 나은 제품을 만들기 위해 노력할 것입니다. 다시 한 번 감사합니다. 당신의 피드백을 기반으로 우리 회사는 더 나은 제품을 만들어 나갈 것입니다.

감사합니다.

[Your Name]
[Your Position]
[Your Contact Information]


In [170]:
test_review = "제품 마감이 너무 잘 되어 있고, 배송도 빨라서 너무 좋아요"
print(f"[고객 리뷰] : {test_review}")

### 자동으로 긍정/부정 라우팅되어 사과문 작성하기
rout_res = safe_branch_chain.invoke({"review" : test_review})
print(f"[답글] : {rout_res}")

[고객 리뷰] : 제품 마감이 너무 잘 되어 있고, 배송도 빨라서 너무 좋아요
routing_chain 결과값 ===>  POSITIVE
[답글] :  감사합니다! 정말 감사합니다. 제품 마감이 매우 신선하고 깨끗하게 되어 있어 매우 기쁩니다. 또한 배송이 빨라서 정말 좋아하는 것 같습니다. 고객님의 만족도를 위해 최선을 다하고 있습니다. 다음에도 좋은 경험을 제공하기 위해 노력하겠습니다. 감사합니다!


In [ ]:
"""
<실습>
 1. 모델 변경 가능하게 처리
 2. 계속 사용자의 질문 받고, 답변 출력 처리
"""

In [193]:
#  1. 모델 변경 가능하게 처리
def change_model(input_data = int(input("숫자를 입력하세요. 1 : mistral or 2 : qwen2.5:0.5b을 "))):
    if input_data == 1:
        input_data = "mistral"
    elif input_data == 2:
        input_data = "qwen2.5:0.5b"
    llm = ChatOllama(
        model = input_data,
        temperature = 0.1,
        max_tokens = 100
    )


change_model()

#  2. 계속 사용자의 질문 받고, 답변 출력 처리
while True:
    test_review = input("리뷰를 입력해주세요. 종료시 종료")
    print(f"[고객 리뷰] : {test_review}")

    ### 자동으로 긍정/부정 라우팅되어 사과문 작성하기
    rout_res = safe_branch_chain.invoke({"review" : test_review})
    print(f"[답글] : {rout_res}")

    if test_review == "종료":
        break

[고객 리뷰] : 너무 좋아요
routing_chain 결과값 ===>  이 리뷰에서는 감정이 표현되지 않으므로, 'POSITIVE' 또는 'NEGATIVE'로 판단하기 어렵습니다. 다음은 리뷰를 해석하여 'POSITIVE'로 판단한 예시입니다.

"This product is amazing! It's so easy to use and the results are fantastic. I highly recommend it to anyone looking for a high-quality product."

이 리뷰에서는 제품이 좋다는 긍정적인 평가와 제품의 결과가 좋다는 평가가 있어 'POSITIVE'로 판단할 수 있습니다.
[답글] :  감사합니다! 당신의 평가와 감사함을 느꼈습니다. 저희 팀에게 매우 중요한 것 중 하나는 고객님들의 만족도를 높이는 것입니다. 당신의 피드백을 통해 우리가 제공하는 서비스를 더욱 개선할 수 있기 때문에 정말 감사합니다. 또한 당신의 다음 방문을 기대합니다! 또한 당신의 친구들에게도 추천해 주시면 감사하겠습니다! 다시 만나서 좋습니다!
[고객 리뷰] : 잘 쓸게요 또 사고 싶어요 !!
routing_chain 결과값 ===>  POSITIVE
[답글] :  감사합니다! 당신의 평가와 감상을 듣고 있어 매우 기쁩니다! 당신이 다음을 다시 쓸 계획이고 더 많은 것을 구매하고 싶어하는 것을 알게 되어 매우 기쁩니다! 당신의 지속적인 지원과 관심 감상에 감사합니다. 앞으로도 더 많은 좋은 작품을 만들어 드리겠습니다! 감사합니다!
[고객 리뷰] : 종료
routing_chain 결과값 ===>  NEGATIVE (The review indicates a negative sentiment as it mentions the game being "boring" and "not worth the money.")
[답글] :  감사합니다 고객님의 피드백을 받아들이고 있습니다. 우리 서비스에 대한 불만이 있어 죄송하게 생각합니다. 우리 팀은 항상 

In [294]:
def route_chain_set():
    request_msg = "다음 리뷰가 긍정적이면 'POSITIVE', 부정적이면 'NEGATIVE'라고 대답해 : {review}"
    input_data = int(input("숫자를 입력하세요. 1 : mistral or 2 : qwen2.5:0.5b을 "))

    if input_data == 1:
        input_data = "mistral"
    elif input_data == 2:
        input_data = "qwen2.5:0.5b"
    llm = ChatOllama(
        model = input_data,
        temperature = 0.1,
        max_tokens = 100
    )
    print(f"사용 모델 {input_data}")
    return (request_msg, llm)



    

In [295]:
i = 0
for n in range(0, 10, 1) :
    i += 1

In [320]:

def prompt_set(p_request_msg, p_llm):
    # - 프롬프트 변수명 : routing_prompt
    routing_prompt = ChatPromptTemplate.from_template(p_request_msg)


    # - 출력 파서 정의
    parser = StrOutputParser()

    global routing_chain
    # - 체인 구성 : 변수명은 routing_chain
    routing_chain = routing_prompt | p_llm | parser

    positive_prompt = ChatPromptTemplate.from_template("매우 기쁜 어조로 다음 리뷰에 대한 감사 답글을 작성해 : {review}")
    # - 체인변수명 : positive_chain
    positive_chain = positive_prompt | p_llm | parser

   
    negative_prompt = ChatPromptTemplate.from_template("매우 죄송한 어조로 다음 리뷰에 대한 사과 답글을 작성해 : {review}")
    # - 체인변수명 : negative_chain
    negative_chain = negative_prompt | p_llm | parser

    return (positive_chain, negative_chain)


def is_positive(x) :
    ### routing_chain을 독립적으로 호출하여 실행
    # - x 매개변수에는 사용자가 질문한 review라는 key의 값이 들어있음
    #   -> x = {"review" : "사용자 질문"}}
    rout = routing_chain.invoke({"review" : x["review"]})

    print(f"routing_chain 결과값 ===> {rout}")
    ### routing_chain의 실행결과가 "POSITIVE" 인지 아닌지 확인 후 반환
    return "POSITIVE" in rout


In [305]:
def safe_branch_chain_set(p_is_positive, p_positive_chain, p_negative_chain):
    branch_chain = RunnableBranch(
        # 조건(True 또는 False)이 참일 때(의도가 긍정일 때) : if
        #  - 조건은 함수(is_positive)로 정의하여 처리 : routing_chain 실행하여 결과값 확인
        #   -> 긍정이면 True, 부정이면 False를 반환해주는 함수 정의하여 사용
        (p_is_positive, p_positive_chain),

        # 조건이 거짓일 때(의도가 부정일 때) : else
        p_negative_chain
    )


    fallback_chain = RunnableLambda(lambda x: "일시적인 오류로 답변을 생성할 수 없습니다. 고객센터로 문의해 주세요!!")

    ### Fallback 상태 확인 함수 정의
    # - 체인 작동 중 예외가 발생했는지 상태확인을 시작하는 함수
    # - 예외가 발생하면 자동으로 fallback_chain을 작동시킴
    safe_branch_chain = branch_chain.with_fallbacks([fallback_chain])

    return safe_branch_chain


def answer(p_safe_branch_chain):
    while True:
        test_review = input("리뷰를 입력해주세요. 종료시 종료")
        print(f"[고객 리뷰] : {test_review}")
        if test_review == "종료":
            break

        ### 자동으로 긍정/부정 라우팅되어 사과문 작성하기
        rout_res = p_safe_branch_chain.invoke({"review" : test_review})
        print(f"[답글] : {rout_res}")

        


In [ ]:

# return 하는 변수명을 기준으로 rs_떙땡 = 함수 라고 변수명을 보통 정함
r_c_s = route_chain_set()

p_s = prompt_set(r_c_s[0], r_c_s[1])

def is_positive(x) :
    ### routing_chain을 독립적으로 호출하여 실행
    # - x 매개변수에는 사용자가 질문한 review라는 key의 값이 들어있음
    #   -> x = {"review" : "사용자 질문"}}
    rout = routing_chain.invoke({"review" : x["review"]})

    print(f"routing_chain 결과값 ===> \n{rout}")
    ### routing_chain의 실행결과가 "POSITIVE" 인지 아닌지 확인 후 반환
    return "POSITIVE" in rout

s_b_c_s = safe_branch_chain_set(is_positive, p_s[0], p_s[1])

answer(s_b_c_s)

사용 모델 mistral
[고객 리뷰] : 너무 좋아요 또 살게요
routing_chain 결과값 ===> 
 POSITIVE
[답글] :  감사합니다! 당신의 감상과 만족을 느꼈어 정말 기쁩니다! 당신의 삶에 더욱 즐거운 시간을 보내기를 바랍니다. 또한 다음에도 당신의 관심과 지원을 기다리고 있습니다. 감사합니다!
[고객 리뷰] : 너무 마감이 별롭니다 다신 안사요
routing_chain 결과값 ===> 
 NEGATIVE (The deadline is too tight, I won't buy again)
[답글] :  감사합니다 고객님의 피드백을 받아들이며 죄송합니다. 당신의 의견을 듣고 있으며 향후 더 나은 서비스를 제공하기 위해 노력하겠습니다. 다음과 같이 사과 답글을 작성해드리겠습니다.

"고객님께서 제공받은 서비스에 대한 피드백 감사합니다. 당신의 의견을 듣고 있으며 서비스의 품질을 향상시키기 위해 노력하겠습니다. 특히 마감이 별롭다는 의견을 들으셨습니다. 이에 대해서는 더 빠르고 정확하게 대처하기 위해 노력하겠습니다. 또한 향후 서비스를 이용하시면 더 나은 경험을 제공하겠습니다. 다시 만나서 좋은 일을 하시길 바랍니다."

이 답글은 고객님의 의견을 듣고 있으며 향후 더 나은 서비스를 제공하기 위해 노력하는 의도를 전달합니다. 또한 죄송하고 사과하는 의미를 담고 있습니다.
[고객 리뷰] : 종료


In [304]:
model_kind = lambda model_name : ChatOllama(model=model_name, temperature=0.1, max_tokens=100)

def chain (model) :
    routing_chain = routing_prompt | model | output_parser
    positive_chain = positive_prompt | model | output_parser
    negative_chain = negative_prompt | model | output_parser
    return routing_chain, positive_chain, negative_chain

while True :
    model_n = input("모델을 선택하여 주세요(번호 입력) : 1. mistral, 2. qwen, 0. 완전종료")    
    if model_n =="0" :
        print("="*50,"종료", "="*50)
        break
    elif model_n == "1" :
        print("="*50, "mistral 선택", "="*50)
        model_name = "mistral"
    elif model_n == "2" :
        print("="*50, "qwen 선택", "="*50)
        model_name = "qwen2.5:0.5b"
    else :
         continue

    model = model_kind(model_name)
    routing_chain, positive_chain, negative_chain = chain(model)
    
    branch_chain = RunnableBranch((is_positive, positive_chain), negative_chain)
    fallback_chain = RunnableLambda(lambda x : "일시적인 오류로 답변을 생성할 수 없습니다. 고객센터로 문의해 주세요!!")
    safe_branch_chain = branch_chain.with_fallbacks([fallback_chain])
    
    while True :
        test_review = input("리뷰를 남겨주세요. 0을 입력하면 모델선택으로 이동.")
        if test_review =="0" :
                print("="*50,"모델선택으로이동", "="*50)
                break
        print(f"[고객 리뷰] : {test_review}")

        ### 자동으로 긍정/부정 라우팅되어 사과문 작성하기
        rout_res = safe_branch_chain.invoke({"review" : test_review})
        print(f"[답글] : {rout_res}")

================================================== mistral 선택 ==================================================
[고객 리뷰] : 너무 좋아요 다음에 또 살게요!
routing_chain 결과값 ===> 
 감사합니다! 정말 감사하게 듣고 있습니다. 저희 팀이 제공하는 서비스를 맘이 맞춰 주셔서 정말 감사합니다. 다음에도 같이 놀아주시면 정말 행복하겠습니다! 또 다시 감사합니다! 앞으로도 좋은 일만 있기를 바랍니다!
[답글] :  감사합니다 고객님의 감상과 평가에 대한 피드백입니다. 정말 감사합니다. 우리 팀은 항상 고객님의 만족도를 최우선으로 생각하고 있습니다. 다음에 또 방문해 주시면 감사하겠습니다. 또한, 다음에 또 살게 되시면 더욱 좋아하시면 하는 것 같네요! 다시 만나서 좋겠습니다. 감사합니다.
================================================== 모델선택으로이동 ==================================================
================================================== 종료 ==================================================
